In [ ]:
# conda create --name sailvl --clone qwen
# conda activate sailvl
# module load cuda/12.2

# pip install timm

In [ ]:
import re
import csv
import glob
import json
import torch
import numpy as np
import torch
import torchvision.transforms as T

from tqdm import tqdm
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def build_transform(input_size):
    MEAN, STD = IMAGENET_MEAN, IMAGENET_STD
    transform = T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=MEAN, std=STD)
    ])
    return transform

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=10, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    # calculate the existing image aspect ratio
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1) for i in range(1, n + 1) for j in range(1, n + 1) if
        i * j <= max_num and i * j >= min_num)
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    # find the closest aspect ratio to the target
    target_aspect_ratio = find_closest_aspect_ratio(
        aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    # calculate the target width and height
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    # resize the image
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        # split the image
        split_img = resized_img.crop(box)
        processed_images.append(split_img)
    assert len(processed_images) == blocks
    if use_thumbnail and len(processed_images) != 1:
        thumbnail_img = image.resize((image_size, image_size))
        processed_images.append(thumbnail_img)
    return processed_images

def load_image(image_file, input_size=448, max_num=10):
    image = Image.open(image_file).convert('RGB')
    transform = build_transform(input_size=input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = [transform(image) for image in images]
    pixel_values = torch.stack(pixel_values)
    return pixel_values

In [ ]:
path = "BytedanceDouyinContent/SAIL-VL-1d6-8B"
model = AutoModel.from_pretrained(
    path,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(path, trust_remote_code=True, use_fast=False)
generation_config = dict(max_new_tokens=1024, do_sample=False)

In [ ]:
text = """You are an image analysis tool specialized in facial attribute classification. 
          For the provided face image, output a JSON object with the following attributes:
        
            {
              "gender": ["male", "female"],
              "age": ["young", "middle-aged", "senior"],
              "skin_color": ["light", "medium", "dark"],
              "ancestry": ["asian", "south_asian", "black", "latino/hispanic", "middle_eastern", "white", "indigenous"],
              "hair_color": ["black", "brown", "red", "blonde", "gray", "other"],
              "bangs": ["yes", "no"],
              "bald": ["yes", "no"],
              "beard": ["no", "mustache", "stubble", "full"],
              "glasses": ["no", "regular", "sun"],
              "headwear": ["no", "beanie", "cap", "hat", "headband", "hijab", "helmet", "turban"],
            }

          Ensure the labeling is based on visible evidence only. If an attribute is unclear, return "unknown".
          
          Only output the JSON without any additional explanation or text.
          
          Example JSON output:
          
          {
            "gender": "female",
            "age": "middle-aged",
            "skin_color": "light",
            "ancestry": "asian",
            "hair_color": "black",
            "bangs": "no",
            "bald": "no",
            "beard": "no",
            "glasses": "sun",
            "headwear": "beanie",
          }
          """

default_keys = ["gender", "age", "skin_color", "ancestry", "hair_color", "bangs", 
                "bald", "beard", "glasses", "headwear"]

# Function to process a single image
def process_images(batch_images):
    # Sample messages for batch inference
    pixel_values = []
    for image_path in batch_images:
        pixel_values.append(load_image(image_path, max_num=12).to(torch.bfloat16).cuda())
    num_patches_list = [pixel_value.size(0) for pixel_value in pixel_values]
    pixel_values = torch.cat(pixel_values, dim=0)
    questions = [f'<image>\n{text}'] * len(num_patches_list)
    
    with torch.inference_mode():
        responses = model.batch_chat(tokenizer, pixel_values,
                                 num_patches_list=num_patches_list,
                                 questions=questions,
                                 generation_config=generation_config)
    result = []
    for i in range(len(batch_images)):
        # Extracting the JSON part
        json_output = re.search(r"\{.*\}", responses[i], re.DOTALL).group()
        json_output = re.sub(r',\s*}', '}', json_output)  # This removes a comma just before a closing curly brace
        data = json.loads(json_output)
        result.append([batch_images[i].replace('../../../data/RFW/data/', '')] + [data.get(key, "unknown") for key in default_keys])
    torch.cuda.empty_cache()
    return result

In [ ]:
init = 0
files = glob.glob('../../../data/RFW/data/*/*/*')

import os
os.makedirs("../../../results/annotations", exist_ok=True)
output_file = "../../../results/annotations/sailvl_labels.csv"

# Column names
columns = ["File Name", "Gender", "Age", "Skin Color", "Ancestry", "Hair Color", 
           "Bangs", "Bald", "Beard", "Glasses", "Headwear"]

# Chunk settings
chunk_size = 32  # Number of images processed per batch

# Write CSV header
with open(output_file, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(columns)  # CSV Header

# Process in chunks
for start_idx in tqdm(range(init, len(files), chunk_size), desc="Processing Images"):
    end_idx = min(start_idx + chunk_size, len(files))
    chunk = files[start_idx:end_idx]
    results = process_images(chunk)
    # Save chunk results to CSV
    with open(output_file, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(results)
    break